# Solutions — merge, join, Indicators, Validation, merge_asof, Reconciliation

Exercise notebook: `09_pandas/05_joins_merge.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

### Setup (same as the exercise notebook)

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
pd.set_option("display.width", 140)
LAB = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets").is_dir() and (p / "labkit").is_dir())   # lab root (works from any folder)
DATA = LAB / "datasets"
orders_raw = pd.read_csv(DATA / "csv" / "orders.csv", dtype=str, keep_default_na=False)
print(orders_raw.shape); orders_raw.head()

## Exercise 1 — 🟢 Easy · Left join with validation

**Working solution**

In [ ]:
def with_customers(orders, customers):
    return orders.merge(customers[["customer_id", "segment", "country"]], on="customer_id", how="left",
                        validate="many_to_one", suffixes=("", "_customer"))

**Explanation**

`validate='many_to_one'` raises `MergeError` if customers has duplicate ids — which would silently **multiply** your orders (and revenue). Always validate joins in pipelines.

**Complexity:** O(n + m).

**Common mistakes**

- Unvalidated join on a dimension with dupes → inflated metrics nobody notices.

## Exercise 2 — 🟡 Medium · Anti-join with indicator

**Working solution**

In [ ]:
def orphans(orders, customers):
    known = customers[["customer_id"]].drop_duplicates()
    merged = orders.merge(known, on="customer_id", how="left", indicator=True)
    mask = (merged["_merge"] == "left_only") & merged["customer_id"].notna() & (merged["customer_id"].astype(str).str.strip() != "")
    return merged.loc[mask, list(orders.columns)].reset_index(drop=True)

**Explanation**

An anti-join (`LEFT JOIN ... WHERE right IS NULL`) via the indicator column. Dedupe the right side's keys first so the merge can't fan out.

**Complexity:** O(n + m).

**Common mistakes**

- `~orders.customer_id.isin(customers.customer_id)` — fine too! But it can't tell you `right_only` rows for reconciliation.

## Exercise 3 — 🟡 Medium · Enrich with multiple dimensions

**Working solution**

In [ ]:
def enrich(orders, customers, products):
    prods = products[["product_id", "category", "name", "price"]].rename(columns={"name": "product_name", "price": "catalog_price"})
    out = (orders.merge(customers[["customer_id", "segment"]], on="customer_id", how="left", validate="many_to_one")
                 .merge(prods, on="product_id", how="left", validate="many_to_one"))
    out["category"] = out["category"].replace("", None).fillna("UNKNOWN")
    out["price_diff_pct"] = ((out["unit_price"] - out["catalog_price"]) / out["catalog_price"] * 100).round(1)
    return out

**Explanation**

Chained merges read top-to-bottom like a star-schema query: fact table, then each dimension. Rename dimension columns *before* merging to avoid suffix soup.

**Complexity:** O(n + c + p).

**Common mistakes**

- `fillna('UNKNOWN')` alone misses empty strings.

## Exercise 4 — 🟠 Hard · As-of joins for FX rates

**Working solution**

In [ ]:
import pandas as pd

def convert_currency(txns, rates):
    left = txns.reset_index().rename(columns={"index": "_orig_index"}).sort_values("ts")
    right = rates.sort_values("effective_ts")
    merged = pd.merge_asof(left, right, left_on="ts", right_on="effective_ts", by="currency", direction="backward")
    merged.loc[merged["currency"] == "INR", "rate_to_inr"] = 1.0
    merged["amount_inr"] = (merged["amount"] * merged["rate_to_inr"]).round(2)
    out = merged.sort_values("_orig_index").set_index("_orig_index")
    out.index.name = txns.index.name
    return out[[*txns.columns, "rate_to_inr", "amount_inr"]]

**Explanation**

`merge_asof` is the vectorized `TreeMap.floorEntry` from Level 4 — both sides must be sorted on the 'on' key. Saving and restoring the original index lets you return rows in caller order.

**Complexity:** O((n + m) log(n + m)).

**Common mistakes**

- Forgetting to sort → `ValueError: left keys must be sorted`.
- Using a regular merge on date and losing intra-day rates.

## Challenge — 🔴 Very Hard · Reconcile two datasets

**Working solution**

In [ ]:
import pandas as pd

def reconcile_frames(source, target, key, columns, tolerance=0.0):
    merged = source[[key, *columns]].merge(target[[key, *columns]], on=key, how="outer", suffixes=("_src", "_tgt"),
                                            indicator=True, validate="one_to_one")
    both = merged.loc[merged["_merge"] == "both"]
    parts = []
    for col in columns:
        s, t = both[f"{col}_src"], both[f"{col}_tgt"]
        if pd.api.types.is_numeric_dtype(s) and pd.api.types.is_numeric_dtype(t):
            differs = (s - t).abs() > tolerance
            differs |= s.isna() ^ t.isna()
        else:
            differs = (s != t) & ~(s.isna() & t.isna())
        if differs.any():
            parts.append(pd.DataFrame({"key": both.loc[differs, key], "column": col,
                                       "source": s[differs].astype(object), "target": t[differs].astype(object)}))
    order = {c: i for i, c in enumerate(columns)}
    mismatches = (pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=["key", "column", "source", "target"]))
    if len(mismatches):
        mismatches = mismatches.sort_values(["key", "column"], key=lambda s: s.map(order) if s.name == "column" else s).reset_index(drop=True)
    counts = merged["_merge"].value_counts()
    summary = {"source_rows": int(len(source)), "target_rows": int(len(target)),
               "only_in_source": int(counts.get("left_only", 0)), "only_in_target": int(counts.get("right_only", 0)),
               "in_both": int(counts.get("both", 0)), "rows_with_mismatch": int(mismatches["key"].nunique()) if len(mismatches) else 0}
    return summary, mismatches

**Explanation**

Outer join + indicator gives presence in one pass; vectorized per-column comparisons give the differences; a long ('tidy') mismatch table is easy to filter, count and send to stakeholders.
`validate="one_to_one"` turns a silent fan-out into a loud error.

**Complexity:** O(n + m) per column.

**Common mistakes**

- `NaN != NaN` is True → every null shows up as a mismatch.
- Comparing floats with `!=`.